<a id="top"></a>

# Demo: wire the ReAct agent graph (on a stub model)

```yaml
title: "Demo: wire the ReAct agent graph (on a stub model)"
keywords: ReAct agent, StateGraph, add_messages, ToolNode, route, back-edge, cycle, conditional edge, stream updates, natural termination, recursion_limit, GraphRecursionError, handle_tool_errors, ToolMessage, FakeModel, offline, teacher demo
estimated duration: 22
```

> **Lesson:** L10. Teacher demo — the runnable spine of the lesson, paired with the
> [L1002 reference lecture](L1002_lecture.md). **No API key needed.** This demo drives the graph with
> a *scripted stub model* (`FakeModel`) so the cycle, the `recursion_limit`, and tool-failure
> handling are **deterministic** — the same run every time. The live multi-step version is
> [L1006](L1006_lecture.ipynb).
>
> The point to land: **an agent is a graph with a cycle.** Two nodes (`agent`, `tools`), a
> conditional exit, and a **back-edge** — the model→tool→model loop, drawn as data.

## Contents

- [1. Setup — the tools and a stub model](#1-setup--the-tools-and-a-stub-model)
- [2. Wire the graph node by node](#2-wire-the-graph-node-by-node)
- [3. Natural termination — stream the cycle](#3-natural-termination--stream-the-cycle)
- [4. The recursion_limit — catching a runaway](#4-the-recursion_limit--catching-a-runaway)
- [5. Tool failure as a message, not a crash](#5-tool-failure-as-a-message-not-a-crash)
- [6. Takeaways](#6-takeaways)

## 1. Setup — the tools and a stub model

Three inline tools — `calculator`, `lookup`, and `flaky_fetch` (used in section 5) — plus the
course's scripted **`FakeModel`** (from `common`), which mimics a LangChain chat model:
`.bind_tools(...)` then `.invoke(messages)` returns the next scripted **`AIMessage`**, whose
`.tool_calls` say which tools it wants run. The graph can't tell the fake from a real
`ChatAnthropic`; only the *script* is fixed. These are the same tools you'll import from
`common/tools.py` in L11 — you meet them inline here first.

In [1]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the whitelist above blocks names/attributes/calls.
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population from a tiny in-memory table, or raise KeyError if missing."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


def flaky_fetch(url: str) -> str:
    """Fetch a URL from a tiny fake network. https://crash raises; other urls return a body."""
    if url == "https://crash":
        raise RuntimeError("connection reset by peer")
    return "the answer is 42"


# The tool list handed to both the model (via bind_tools) and the ToolNode. LangChain/LangGraph
# infer each tool's schema from its type hints + docstring, so a plain typed function IS the schema.
TOOLS = [calculator, lookup, flaky_fetch]
print("tools:", [t.__name__ for t in TOOLS])

tools: ['calculator', 'lookup', 'flaky_fetch']


[↑ Back to top](#top)

## 2. Wire the graph node by node

The whole agent is **two nodes and two edges**. We build it in a `build_agent` helper so sections
3–5 can reuse it unchanged (the roadmap's Demo 1 graph, stretched and broken later).

1. **State** — a `TypedDict` with `messages: Annotated[list, add_messages]`. The `add_messages`
   **reducer** *appends* each node's messages to the conversation (in L04 a returned key *overwrote*
   it) — that's why the history grows every turn.
2. **`agent` node** — one line: `model.bind_tools(TOOLS).invoke(state["messages"])`, wrapped in the
   `messages` key. Returns one `AIMessage`, possibly carrying `.tool_calls`.
3. **`route`** — the L05 conditional edge: `"tools"` if the last message has `.tool_calls`, else
   `END`. (In L11 you'll meet its prebuilt twin, `tools_condition`.)
4. **Wiring** — `ToolNode(TOOLS)` is the `tools` node; `add_conditional_edges("agent", route, …)` is
   the exit; `add_edge("tools", "agent")` is the **back-edge** that makes it a cycle.

In [2]:
class AgentState(TypedDict):
    """The agent's state: one growing message list. `add_messages` APPENDS, never overwrites."""

    messages: Annotated[list[BaseMessage], add_messages]


def build_agent(model: Any, *, handle_tool_errors: bool = True) -> Any:
    """Wire the ReAct graph: agent node + ToolNode + conditional route + back-edge, compiled.

    `model` is any bind_tools-capable chat model (a FakeModel here, a real ChatAnthropic in
    L1006). `handle_tool_errors` is passed straight to ToolNode — section 5 toggles it.
    """

    def agent_node(state: AgentState) -> dict[str, list[BaseMessage]]:
        """Call the model on the running conversation; return its one reply to be appended."""
        reply = model.bind_tools(TOOLS).invoke(state["messages"])
        return {"messages": [reply]}

    def route(state: AgentState) -> str:
        """Conditional edge: got tool calls? -> the tools node; else -> END (natural stop)."""
        last = state["messages"][-1]
        if isinstance(last, AIMessage) and last.tool_calls:
            return "tools"
        return END

    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS, handle_tool_errors=handle_tool_errors))
    builder.set_entry_point("agent")
    # The conditional exit (route) and the plain back-edge -- the only two edges in the graph.
    builder.add_conditional_edges("agent", route, {"tools": "tools", END: END})
    builder.add_edge("tools", "agent")
    return builder.compile()

Render what we just wired. `draw_mermaid()` prints the graph offline (no network): an `agent` node,
a `tools` node, the **back-edge** `tools → agent`, and the conditional exit out of `agent`. This is
the picture from the [L1002 lecture](L1002_lecture.md), read straight off the compiled graph.

In [3]:
demo_graph = build_agent(FakeModel([text_reply("placeholder")]))
rendered = demo_graph.get_graph()
print(rendered.draw_mermaid())

# The nodes and the all-important back-edge, read off the compiled graph:
print("nodes:", list(rendered.nodes))
edges = [(e.source, e.target) for e in rendered.edges]
print("back-edge tools -> agent present:", ("tools", "agent") in edges)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

nodes: ['__start__', 'agent', 'tools', '__end__']
back-edge tools -> agent present: True


[↑ Back to top](#top)

## 3. Natural termination — stream the cycle

Script a model that calls `calculator`, then (seeing the result) calls `lookup`, then answers in
plain text. Two tool-call turns, then a final message with **no `.tool_calls`** — that is *natural
termination*, `route` returning `END`.

We watch the cycle turn with `graph.stream(task, stream_mode="updates")` — the same run-inspection
call you've used since L03 — which yields **one chunk per node firing**: `agent` → `tools` → `agent`
→ `tools` → `agent`. Name each time control crossed the back-edge.

In [4]:
happy_script = [
    tool_reply(tool_call("c1", "calculator", {"expression": "17*17 - 1"})),  # -> 288
    tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),  # -> 37000000
    text_reply("The puzzle city is Tokyo; its population is about 37,000,000."),
]
graph = build_agent(FakeModel(happy_script))

task = {
    "messages": [HumanMessage(content="Population of the city that is 17*17 - 1? Use the tools.")]
}
for chunk in graph.stream(task, stream_mode="updates"):
    for node_name, update in chunk.items():
        new_messages = [type(m).__name__ for m in update["messages"]]
        print(f"[node: {node_name:5}] appended {new_messages}")

[node: agent] appended ['AIMessage']
[node: tools] appended ['ToolMessage']
[node: agent] appended ['AIMessage']
[node: tools] appended ['ToolMessage']
[node: agent] appended ['AIMessage']


`invoke` hands back only the *final* state, so you can also just read the whole `messages` list and
narrate the run turn by turn — the **minimum-viable trace** L12 will replace with something
structured.

In [5]:
def describe(msg: BaseMessage) -> str:
    """One readable line per message: type, any tool calls, tool results, or final text."""
    kind = type(msg).__name__
    if isinstance(msg, AIMessage) and msg.tool_calls:
        calls = ", ".join(f"{c['name']}({c['args']})" for c in msg.tool_calls)
        return f"{kind:12} -> tool call: {calls}"
    if isinstance(msg, ToolMessage):
        return f"{kind:12} <- result [{msg.status}]: {msg.content!r}"
    return f"{kind:12}    {str(msg.content)!r}"


final = build_agent(FakeModel(happy_script)).invoke(task)
for msg in final["messages"]:
    print(describe(msg))
print("\nfinal answer:", final["messages"][-1].content)

HumanMessage    'Population of the city that is 17*17 - 1? Use the tools.'
AIMessage    -> tool call: calculator({'expression': '17*17 - 1'})
ToolMessage  <- result [success]: '288'
AIMessage    -> tool call: lookup({'city': 'Tokyo'})
ToolMessage  <- result [success]: '37000000'
AIMessage       'The puzzle city is Tokyo; its population is about 37,000,000.'

final answer: The puzzle city is Tokyo; its population is about 37,000,000.


[↑ Back to top](#top)

## 4. The `recursion_limit` — catching a runaway

Now a model that **never stops**: it keeps asking for `lookup`. With a real model this is the
classic runaway. We script several *distinct* `lookup` turns so each is appended (the `add_messages`
reducer pairs messages by id — repeating the *same* object would be de-duplicated, so a runaway
needs fresh turns). The graph's built-in **`recursion_limit`** — set on `invoke`, counting node
visits — halts it with a `GraphRecursionError`.

**Hitting the cap is always a signal worth investigating**, not noise.

In [6]:
from langgraph.errors import GraphRecursionError

# Distinct turns (r0, r1, ...) so add_messages appends each one and the cycle actually runs away.
runaway_script = [tool_reply(tool_call(f"r{i}", "lookup", {"city": "Tokyo"})) for i in range(8)]
runaway_graph = build_agent(FakeModel(runaway_script))

try:
    runaway_graph.invoke(
        {"messages": [HumanMessage(content="Keep re-checking Tokyo forever.")]},
        {"recursion_limit": 6},  # halt after 6 node visits (~3 cycles)
    )
    print("no error (unexpected)")
except GraphRecursionError:
    print(
        "GraphRecursionError: recursion_limit=6 fired -- the cap caught the runaway, not the model"
    )

GraphRecursionError: recursion_limit=6 fired -- the cap caught the runaway, not the model


[↑ Back to top](#top)

## 5. Tool failure as a message, not a crash

`flaky_fetch("https://crash")` **raises** a `RuntimeError`. The `tools` node is a `ToolNode`, and
its `handle_tool_errors` argument decides what happens:

- **`handle_tool_errors=False`** — the exception escapes the node and **kills the whole `invoke`.**
  One buggy tool crashes the agent.
- **`handle_tool_errors=True`** — `ToolNode` catches it and appends a `ToolMessage(status="error")`,
  so the **back-edge** hands the model a message; it sees the error and can recover.

One argument, crash vs. recover.

In [7]:
# handle_tool_errors=False: the raise escapes and crashes the graph.
crash_script = [
    tool_reply(tool_call("x1", "flaky_fetch", {"url": "https://crash"})),
    text_reply("unreachable — the graph already crashed"),
]
strict_graph = build_agent(FakeModel(crash_script), handle_tool_errors=False)
try:
    strict_graph.invoke({"messages": [HumanMessage(content="Fetch https://crash")]})
    print("no error (unexpected)")
except RuntimeError as exc:
    print(f"handle_tool_errors=False -> the exception escaped: {exc!r}")

handle_tool_errors=False -> the exception escaped: RuntimeError('connection reset by peer')


In [8]:
# handle_tool_errors=True: the raise becomes a ToolMessage(status="error"); the model recovers.
recover_script = [
    tool_reply(tool_call("x1", "flaky_fetch", {"url": "https://crash"})),  # raises -> error message
    tool_reply(tool_call("x2", "flaky_fetch", {"url": "https://ok"})),  # retry a good url
    text_reply("https://crash failed, so I fetched https://ok: the answer is 42."),
]
safe_graph = build_agent(FakeModel(recover_script), handle_tool_errors=True)
recovered = safe_graph.invoke(
    {"messages": [HumanMessage(content="Fetch https://crash, else retry https://ok")]}
)
for msg in recovered["messages"]:
    print(describe(msg))

HumanMessage    'Fetch https://crash, else retry https://ok'
AIMessage    -> tool call: flaky_fetch({'url': 'https://crash'})
ToolMessage  <- result [error]: "Error: RuntimeError('connection reset by peer')\n Please fix your mistakes."
AIMessage    -> tool call: flaky_fetch({'url': 'https://ok'})
ToolMessage  <- result [success]: 'the answer is 42'
AIMessage       'https://crash failed, so I fetched https://ok: the answer is 42.'


[↑ Back to top](#top)

## 6. Takeaways

- **An agent is a graph with a cycle.** The same `build_agent` produced every run above; only the
  *script* changed. Two nodes (`agent`, `tools`), a conditional exit, and a **back-edge** — the
  model→tool→model loop, drawn as data. (In [L1006](L1006_lecture.ipynb) only the *model* changes —
  a real Claude replaces the stub, graph unchanged.)
- **The reducer + `ToolNode` maintain the message-history invariant.** `add_messages` appends;
  `ToolNode` answers each `.tool_calls` entry with a matching `ToolMessage`. You didn't write the
  bookkeeping that is the #1 bug in a hand loop.
- **Termination is a branch of `route`.** Natural termination = `route` returns `END` (no tool
  calls); `recursion_limit` is the safety cap for everything else. A cyclic graph with no cap is a
  runaway waiting to happen.
- **Tool failures are messages, not exceptions.** `ToolNode(handle_tool_errors=True)` turns a raise
  into a `ToolMessage(status="error")` the back-edge hands back — the model decides whether to retry
  (reinforces [L08](../L08/objectives.md)'s error thinking at the *graph* layer).
- Next: build the graph yourself in [L1004](L1004_lab_empty.ipynb) (wiring + termination) and
  [L1005](L1005_lab_empty.ipynb) (tool failures), then watch it drive a real model in
  [L1006](L1006_lecture.ipynb).

[↑ Back to top](#top)